# Plot PPSD

Historical analysis; scientific assumptions require review.

Source: `taylor-kenyon/FluvialSeismology_Analysis`, `PPSD.ipynb`, commit `2764f9454916b0162f98b7221883d75ca742c646`. Calculations are preserved; imports and paths have been adapted. Stored source outputs have been cleared.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "project_paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter inside the mora-fluvial-analysis repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from project_paths import method_path, result_path
METHOD_ROOT = PROJECT_ROOT / 'fluvial_seismology'
os.chdir(METHOD_ROOT)


# Get Data + RSAM Plots + Spectrograms + PPSDs

In [ ]:
#Working from documentation for Visualizing Probabalistic Power Spectra Densities in the OBSPY 1.4.1 user guide.
#Objective is to determine a strategy for assessing signal quality between different sites and times.
#Signals validated by whatever tool can be produced here are intended to be of enough quality to carry forward for figures and additional analysis

#st = read("https://examples.obspy.org/BW.KW1..EHZ.D.2011.037")
#tr = st.select(id="BW.KW1..EHZ")[0]

import obspy
from obspy import read, UTCDateTime
from obspy.io.xseed import Parser
from obspy.signal import PPSD
import numpy as np
import matplotlib.pyplot as plt
from obspy.imaging.cm import pqlx
import os
from obspy.clients.fdsn import Client
from obspy.clients import fdsn
from fluvial_seismology.scripts.processing import waveform_utils as waveformUtils
from fractions import Fraction
import importlib
importlib.reload(waveformUtils)

In [ ]:
# setting up procedure

# define start and end time for the data range
start_date = UTCDateTime(2023, 8, 20)  # Start time
end_date = UTCDateTime(2023, 8, 24)  # End time
delta = 86400  # 1 day in seconds
stas = [2301, 2302, 2303, 2304, 2305, 2306, 2307, 2308, 2309, 2310, 2311, 2312, 2313, 2314, 2315, 2316,
        'G2301', 'G2302', 'G2303', 'G2304', 'G2305', 'G2306', 'G2307', 'G2308', 'G2309', 'G2310', 'G2311',
        'G2312', 'G2313', 'G2314', 'G2315', 'G2316']
stas = ['ZE.2303..GPZ']  # Station IDs

# set station info
nowsta = stas[0]
network, station, location, channel = nowsta.split('.')
print(f"Using station: {nowsta}")

# define path where .mseed files are stored
path = method_path('fluvial_seismology', 'data/raw/seismic_data/NO2303/GPZ')

# check if the path exists, create if not
if not os.path.exists(path):
    print(f"Path '{path}' does not exist. Creating directory...")
    os.makedirs(path, exist_ok=True)
else:
    print(f"Path '{path}' already exists.")

file_pattern = f'{path}/{network}.{station}.*.{channel}*.mseed'

# set up IRIS client
DATASELECT = 'http://service.iris.edu/ph5ws/dataselect/1'
c = fdsn.client.Client(
    service_mappings={
        'dataselect': DATASELECT,
    },
)

# IRIS credentials
username = os.environ.get("MORA_IRIS_USERNAME")
password = os.environ.get("MORA_IRIS_PASSWORD")
if bool(username) != bool(password):
    raise ValueError("Set both MORA_IRIS_USERNAME and MORA_IRIS_PASSWORD, or neither.")
if username:
    c.set_credentials(username, password)
# set up metadata
client = Client("IRISPH5", timeout=600)
#inv = client.get_stations(network='ZE', station=stas[0], channel='GPZ', level='response')  # adjust as needed
inv = client.get_stations(network=network, station=station, location=location, channel=channel, level='response')

In [ ]:
# function to get data from IRIS
def get_iris_data(t1, t2, stas, path):
    """Function to get data from IRIS is don't already have"""
    tNow = t1  # start time
    extract_delta = 1800 # 30 mins in seconds

    while tNow < t2:
        #print(f"Retrieving data from IRIS for {tNow} to {tNow + extract_delta}")
        for nowsta in stas:
            print(f"Retrieving IRIS data for station {nowsta} from {tNow} to {tNow + extract_delta}")

            try:
                #S = c.get_waveforms('ZE', nowsta, '', 'GPZ', tNow, tNow + extract_delta)
                S = c.get_waveforms(network, station, location, channel, tNow, tNow + extract_delta) # try adding 1 second to end for spectra? prob only works if >1 day timeframe..
                print(f"Data for {nowsta} retrieved successfully.")

                # resample the data
                print('Resampling data...')
                S.resample(350)  # resample to 350 Hz/s

                # save the data to .mseed files
                print('Saving data to .mseed files...')
                for tr in S:
                    filename = os.path.join(path, f'{tr.id}_{tr.stats.starttime.strftime("%Y%m%d%H%M%S")}.mseed') # os.path.join() should work for Windows and Macs, optionally add: .replace("\\","/")
                    #filename = '{}/{}_{}.mseed'.format(path, tr.id, tr.stats.starttime.strftime('%Y%m%d%H%M%S'))
                    Path(filename).parent.mkdir(parents=True, exist_ok=True)
                    tr.write(filename)
                    print(f"Saved {filename}")
            except Exception as e:
                print(f"Error retrieving data for {nowsta} from {tNow} to {tNow + extract_delta}: {e}")

        tNow += extract_delta  # move to the next day

    print('Data retrieval and saving completed.')

# call function independently if only want data
#get_iris_data(t1,t2,stas,path)

In [ ]:
# function to plot spectrogram
def plot_spectrogram(S, t1, t2):
    """Function to plot RSAM and Spectrogram"""
    S.merge(method=1) # merge traces to avoid duplicates

    f1 = waveformUtils.multiDaySpectrogram(S, averageLength=3600, fftLength=60, minFreq=0.5, maxFreq=150, cmap='turbo',
                                           dateLimits=None, vmin=0.4, vmax=0.9, plotAverage=True)

    # if just running this cell, comment out f1.show() and plt.show() to avoid displaying twice
    f1.show()
    plt.show() # won't display each loop in main function without this line
    plt.close()
    return f1

# call function independently - uncomment below and comment out .show()s ^
# end=end_date+(6*60)
# S = obspy.read(file_pattern, starttime=start_date, endtime=end)
# plot_spectrogram(S,t1=start_date,t2=end)

In [ ]:
# function to plot ppsd and temporal plot
def plot_ppsd(S, inv, t1, t2, user_values):
    """Function to calculate and plot PPSD and Temporal Plot"""
    S.merge(method=1) # merge traces to avoid duplicates
    S.resample(300) # resample

    # initialize PPSD object
    ppsd = None
    for tr in S:
        if ppsd is None:
            ppsd = PPSD(tr.stats, metadata=inv, period_limits=(1/150, 10.0))
            # this format possibly avoids re-iterating over individual traces/re-creating PPSD object within loop
            #ppsd = PPSD(S[0].stats, metadata=inv, period_limits=(1/150, 10.0))  # adjust period limits

        ppsd.add(tr)
        print('Trace added to PPSD')

    # plot PPSD for the current day
    if ppsd is not None and len(ppsd.times_processed) > 0:
        print(f"Data accumulated for {(t2-t1)/(60*60*24)}-day periods starting on {t1}")
        ppsd.plot(period_lim=(0.1,150),cmap=pqlx,xaxis_frequency=True) # if xaxis_frequency=False, set period_lim=(1/150,10)
        plt.close() # prevents from displaying each iteration

        # plot temporal plot if user values are provided
        if user_values:
            print(f"Plotting temporal plot for periods: {user_values}")
            ppsd.plot_temporal(user_values) # plot temporal plot with user values
            plt.show()
            plt.close()

        # save the plot as .png
        #plot_filename = f"ppsd_plot_{t1.date}.png" # change title if more than one date
        #plt.savefig(plot_filename)
        #print(f"Plot saved as {plot_filename}")

    else:
        print(f"No PPSD data accumulated for {t1} to {t2}")

    return ppsd

# # call function independently - uncomment below
# # likely wouldn't to run this without spectra though since want to pick what frequency bins for temporal plot

# # user input frequency values
# user_inputs = input("Enter frequency values for temporal plot (comma-separated, takes in fractions and decimals): ").split(',')

# # convert fractions and regular floats
# user_freqs = []
# user_periods = []
# for freq in user_inputs:
#     try:
#         freq = freq.strip() # clean up extra spaces
#         if '/' in freq:
#             user_freqs.append(float(Fraction(freq))) # handle fraction
#         else:
#             user_freqs.append(float(freq)) # handle float
#     except ValueError:
#         print(f"Invalid input: {freq}. Using default values.")

# # convert frequencies to periods
# user_periods = [1 / freq for freq in user_freqs]  # converts to periods for temporal plot
# print(f'User-defined periods: {user_periods}')

# user_periods = [1 / freq for freq in user_freqs] # converts to periods for temporal plot
# print('periods:', user_periods)
# print('freqs:', user_freqs)

# # if user didn't provide any valid values, set defaults
# if not user_freqs:
#     print("No valid input detected. Using default frequency values.")
#     user_freqs = [100, 10, 1]  # default frequencies in Hz

# round_periods = ['%.4f' % per for per in user_periods]
# print(f"Inputted frequencies: {user_freqs} --> periods: {round_periods}") # for checking
# #print(user_values) # for checking

# # call plot_ppsd function with user-provided values
# plot_ppsd(S, inv, t1=start_date, t2=end_date, user_values=user_periods)
# #plot_ppsd(S, inv, t1, t2, user_values)

In [ ]:
# main function to process data
def process_data(start_date, end_date, stas, path):
    """Main function to run through each day of data in timeframe and perform analyses"""
    delta = 86400  # 1 day in seconds
    current_date = start_date

    while current_date < end_date:
        t1 = current_date
        t2 = current_date + delta  # the next day
        print(f"\nProcessing data for {t1} to {t2}")

        # check if data exists locally for the current date range
   ###  #file_pattern = f'{path}/ZE.*.*.mseed'  # adjust this based on naming convention
        #file_pattern = f'{path}/{network}.{station}.*.{channel}*.mseed' ## if this version, need .split('.')

        try:
            # reading local data for the current date range
            S = obspy.read(file_pattern, starttime=t1, endtime=t2)

            if len(S) == 0:
                print(f"No local data found for {t1} to {t2}. Proceeding to fetch from IRIS.")
                get_iris_data(t1,t2,stas,path)
                S = obspy.read(file_pattern, starttime=t1, endtime=t2)

            else: # if len(S) > 0, some of the stream is stored locally
                loc_start = S[0].stats.starttime
                loc_end = S[-1].stats.endtime

                # if locally stored data doesn't cover the full timerange --> fetch from IRIS
                if loc_start > t1 and loc_end < t2:  # if data is missing at both the beginning and end
                    print(f"Local data is missing, fetching data from {t1} to {t2}")
                    get_iris_data(t1, t2, stas, path)
                    S = obspy.read(file_pattern, starttime=t1, endtime=t2)  # re-read data after fetching
                elif loc_start > t1: # data missing at the beginning
                    print(f"Local data is missing, fetching data from {t1} to {loc_start}")
                    get_iris_data(t1, loc_start, stas, path)
                    S = obspy.read(file_pattern, starttime=t1, endtime=t2) # re-read data after fetching
                elif loc_end < t2: # data missing at the end
                    print(f"Local data is missing, fetching data from {loc_end} to {t2}")
                    get_iris_data(loc_end, t2, stas, path)
                    S = obspy.read(file_pattern, starttime=t1, endtime=t2) # re-read data after fetching
                else: # data missing in somewhere in timeframe, loc_start <= t1 and loc_end >= t2
                    print(f"Local data covers full time range from {t1} to {t2}, proceeding with analysis")

            # plot spectrogram
            S1 = S.copy()  # create a copy of original stream

            # read an extra 6 mins of data to extend the stream
            spec_end = t2 + (6*60) # 6 minutes beyond original end time
            S_ext = obspy.read(file_pattern, starttime=t2, endtime=spec_end) # read only the extra data

            # merge extra 6 mins of data with copied stream (now whole time range is 6 mins longer)
            S1.extend(S_ext)

            # Now S_extended contains the original data plus the extra 6 minutes, and S is untouched
            f1 = plot_spectrogram(S1, t1, spec_end)  # Pass the extended stream to the plot_spectrogram function

            # ask user for frequency values after spectrogram
            print("Enter frequency values for temporal plot (comma-separated, takes in fractions and decimals):")
            user_inputs = input("Example (0.5, 50, 1/100) : ").split(',')

            # process user input - converts fractions and regular floats
            user_freqs = []
            user_periods = []
            for freq in user_inputs:
                try:
                    freq = freq.strip()  # clean up extra spaces
                    if '/' in freq:
                        user_freqs.append(float(Fraction(freq))) # handle fraction
                    else:
                        user_freqs.append(float(freq)) # handle float
                except ValueError:
                    print(f"Invalid input: {freq}. Using default values.")

            user_periods = [1 / freq for freq in user_freqs] # converts to periods for temporal plot
            print('periods:', user_periods)
            print('freqs:', user_freqs)

            # if user didn't provide any valid values, set defaults
            if not user_freqs:
                print("No valid input detected. Using default frequency values.")
                user_freqs = [100, 10, 1]  # default frequencies in Hz

            # plot PPSD
            #plot_ppsd(S, inv, t1, t2, user_values)
            plot_ppsd(S, inv, t1, t2, user_periods)

            round_periods = ['%.4f' % per for per in user_periods]
            print(f"Inputted frequencies: {user_freqs} --> periods: {round_periods}") # for checking
            #print(user_values) # for checking

        except Exception as e:
            print(f"An error occurred while processing data from {t1} to {t2}: {e}")

        current_date += delta # move to next day

# call function
process_data(start_date, end_date, stas, path)
print(f"Analysis for {stas} from {start_date} to {end_date} complete.")

### example with ZE 2303 GPZ

In [ ]:
# older version, but contains printed plots for quicker reference, will delete when above is finalized

## this is run for ZE 2303 GPZ 8/11 - 8/26 (15 days)
# takes ~3.5 mins for each set of plots to display

## whole date range for ZE 2302 is 2023-08-10 to 2023-09-09
## whole date range for ZE 2303 is 2023-07-12 to 2023-09-08
## whole date range for ZE 2304 is 2023-07-12 to 2023-09-07
# exclude first and last day for cleaner data

# define start and end time for the data range
start_date = UTCDateTime(2023, 8, 11)  # Start time
end_date = UTCDateTime(2023, 8, 26)  # End time
delta = 86400
stas = [2301, 2302, 2303, 2304, 2305, 2306, 2307, 2308, 2309, 2310, 2311, 2312, 2313, 2314, 2315, 2316,
        'G2301', 'G2302', 'G2303', 'G2304', 'G2305', 'G2306', 'G2307', 'G2308', 'G2309', 'G2310', 'G2311',
        'G2312', 'G2313', 'G2314', 'G2315', 'G2316']

# define path where .mseed files are stored
path = method_path('fluvial_seismology', 'data/raw/seismic_data/NO2303/GPZ')
stas = ['ZE.2303..GPZ']  # station IDs
if not os.path.exists(path):
    os.makedirs(path, exist_ok=True)

# set station info
nowsta = stas[0]
network, station, location, channel = nowsta.split('.')
print(f"Station: {nowsta}")

# define the file pattern for reading mseed files
# assuming filenames follow the pattern 'ZE.2304..GPZ_YYYYMMDDTHHMMSS.mseed'
file_pattern = f'{path}/{network}.{station}.*.{channel}*.mseed'

# set up metadata
client = Client("IRISPH5", timeout=600) # timeout (s) - time before connection timeout is raised, default 10s
inv = client.get_stations(network=network, station=station, location=location, channel=channel, level='response')

# loop through each day in the date range
current_date = start_date
while current_date < end_date:
    t1 = current_date
    t2 = current_date + delta  # the next day
    print(f"\nProcessing data for {t1} to {t2}")

    try: # read data for the current day
        S = obspy.read(file_pattern, starttime=t1, endtime=t2)
        ## RSAM plot before resample?
    except Exception as e:
        print(f"Error reading data for {t1} to {t2}: {e}")
        current_date += delta
        continue  # skip to next date if there's an error reading data

    # merge traces to avoid duplicates
    S.merge(method=1)

    # or plot RSAM after merge, before resample?
    f1 = waveformUtils.multiDaySpectrogram(S, averageLength=3600, fftLength=60, minFreq=0.5, maxFreq=150, cmap='turbo',
                                      dateLimits=None,
                                      vmin=0.4, vmax=0.9, plotAverage=True)
    f1.show()

    # resample the data
    print('Resampling...')
    S.resample(300)  # Resampled rate for final analysis. Note: Consider Nyquist criterion

    # or plot RSAM after resample?

    # create PPSD object and add traces
    ppsd = None
    for tr in S:
        if ppsd is None:
            ppsd = PPSD(tr.stats, metadata=inv, period_limits=(1/150, 10.0)) # ppsd computed for 0.1 to 200 Hz --> should this be 0.0057s (175 Hz) since new resample rate??
        ppsd.add(tr)
        print('Trace added')

    # plot the PPSD for the current day
    if ppsd is not None and len(ppsd.times_processed) > 0:
        print(f"Data accumulated for {(t2-t1)/(60*60*24)}-day periods starting on {t1}")
        #print("Processed times:", ppsd.times_processed[:10])  # see first 10 times processed

        # plot the PPSD
        plt.figure(figsize=(10, 6))
        ppsd.plot(period_lim=(1/150,10),cmap=pqlx) # to zoom into a specific range, adjust period_lim
        psd_values = ppsd.psd_values
        print("Shape of psd values: ", np.shape(psd_values))
        #print(psd_values[0:1]) # first branch

        # add x-axis of PST

        # plotting time series of PSD values
        ppsd.plot_temporal([0.1,1,10])
        # use these values instead:
        #ppsd.plot_temporal([0.00667,0.01,0.1]) # period bins (s)

        # save the plot as .png
        #plot_filename = f"ppsd_plot_{t1.date}.png" # change title if more than one date
        #plt.savefig(plot_filename)
        #print(f"Plot saved as {plot_filename}")

        plt.close()  # close plot to avoid displaying it in each iteration

    else:
        print(f"No data accumulated for {t1} to {t2}, check if exists")

    # move to the next day
    current_date += delta



In [ ]:
print("the following cells are the same 5 day window of 2302, looking at different frequency bins")

In [ ]:
# look at and determine frequency bins
# use temporal plots

# define start and end time for the data range
start_date = UTCDateTime(2023, 8, 12)  # Start time
end_date = UTCDateTime(2023, 8, 17)  # End time
delta = 86400
stas = [2301, 2302, 2303, 2304, 2305, 2306, 2307, 2308, 2309, 2310, 2311, 2312, 2313, 2314, 2315, 2316,
        'G2301', 'G2302', 'G2303', 'G2304', 'G2305', 'G2306', 'G2307', 'G2308', 'G2309', 'G2310', 'G2311',
        'G2312', 'G2313', 'G2314', 'G2315', 'G2316']

# define path where .mseed files are stored
path = method_path('fluvial_seismology', 'data/raw/seismic_data/NO2302/GPZ')
stas = ['ZE.2302..GPZ']  # station IDs
if not os.path.exists(path):
    os.makedirs(path, exist_ok=True)

# set station info
nowsta = stas[0]
network, station, location, channel = nowsta.split('.')
print(f"Station: {nowsta}")

# define the file pattern for reading mseed files
# assuming filenames follow the pattern 'ZE.2304..GPZ_YYYYMMDDTHHMMSS.mseed'
file_pattern = f'{path}/{network}.{station}.*.{channel}*.mseed'

# set up metadata
client = Client("IRISPH5", timeout=600) # timeout (s) - time before connection timeout is raised, default 10s
inv = client.get_stations(network=network, station=station, location=location, channel=channel, level='response')

# NRL configuration for sensor
sensors = {
    'geophone': 'sensor_DTCC_DT-SOLO_LF5_RC1850_RS43000_SG76.6_STgroundVel'  # SmartSolo sensor configuration
    # Additional sensors can be added to this dictionary as needed
}
# dictionary of EPIC loggers
loggers = {'smartsolo' : 'datalogger_DTCC_SmartSolo-IGU-16HR3C_PD18_FR1000_FPLP_DFOff', # logger for SmartSolo
           # Additional loggers can be added to this dictionary as needed
          }

# loop through each day in the date range
current_date = start_date
while current_date < end_date:
    t1 = current_date
    t2 = current_date + delta  # the next day
    print(f"\nProcessing data for {t1} to {t2}")

    try: # read data for the current day
        S = obspy.read(file_pattern, starttime=t1, endtime=t2)
        ## RSAM plot before resample?
    except Exception as e:
        print(f"Error reading data for {t1} to {t2}: {e}")
        current_date += delta
        continue  # skip to next date if there's an error reading data

    # merge traces to avoid duplicates
    S.merge(method=1)

    # or plot RSAM after merge, before resample?
    f1 = waveformUtils.multiDaySpectrogram(S, averageLength=3600, fftLength=60, minFreq=0.5, maxFreq=150, cmap='turbo',
                                      dateLimits=None,
                                      vmin=0.4, vmax=0.9, plotAverage=True)
    f1.show()

    #response_z = read_inventory(method_path('fluvial_seismology', 'data/raw/instrument_metadata/ph5-station_ZE2303GPZ.xml'))  # vertical component xml # creates an ObsPy Inventory object containing response information
    #print("Add filter...")
    #S.filter("highpass", freq=0.002).remove_response(inventory=response_z)

    # # prefiltering for instrument's frequency bandwidth, do before preprocessing
    # for tr in S: # adjust to not read below 0.1 Hz***
    #     pre_filt = [0.1, 1, 100, 150] # values TBD # 4 corner frequencies
    #     tr.remove_response(inventory=inv, pre_filt=pre_filt, plot=True) # , output="DISP", water_level=60, plot=True)

    # resample the data
    print('Resampling...')
    S.resample(175)  # Resampled rate for final analysis. Note: Consider Nyquist criterion
    ## should resample be 2*(max freq of interest) = 2*150 OR 0.5*(initial data save sample rate) = 0.5*350 ?

    # or plot RSAM after resample?

    # create PPSD object and add traces
    ppsd = None
    for tr in S:
        if ppsd is None:
            ppsd = PPSD(tr.stats, metadata=inv, period_limits=(0.0057, 10.0)) # ppsd computed for 0.1 to 200 Hz --> should this be 0.0057s (175 Hz) since new resample rate??
        ppsd.add(tr)
        print('Trace added')

    # plot the PPSD for the current day
    if ppsd is not None and len(ppsd.times_processed) > 0:
        print(f"Data accumulated for {(t2-t1)/(60*60*24)}-day periods starting on {t1}")
        #print("Processed times:", ppsd.times_processed[:10])  # see first 10 times processed

        # plot the PPSD
        plt.figure(figsize=(10, 6))
        ppsd.plot(period_lim=(0.0057,10),cmap=pqlx) # to zoom into a specific range, adjust period_lim
        psd_values = ppsd.psd_values
        print("Shape of psd values: ", np.shape(psd_values))
        #print(psd_values[0:1]) # first branch

        # add x-axis of PST

        print("bins: [0.007, 0.01, 1] s = ~[140, 100, 1] Hz")
        # plotting time series of PSD values
        ppsd.plot_temporal([0.007,0.01,1])
        # use these values instead:
        #ppsd.plot_temporal([0.00667,0.01,0.1]) # period bins (s)

        # save the plot as .png
        #plot_filename = f"ppsd_plot_{t1.date}.png" # change title if more than one date
        #plt.savefig(plot_filename)
        #print(f"Plot saved as {plot_filename}")

        plt.close()  # close plot to avoid displaying it in each iteration

    else:
        print(f"No data accumulated for {t1} to {t2}, check if exists")

    # move to the next day
    current_date += delta





In [ ]:
# define start and end time for the data range
start_date = UTCDateTime(2023, 8, 12)  # Start time
end_date = UTCDateTime(2023, 8, 17)  # End time
delta = 86400
stas = [2301, 2302, 2303, 2304, 2305, 2306, 2307, 2308, 2309, 2310, 2311, 2312, 2313, 2314, 2315, 2316,
        'G2301', 'G2302', 'G2303', 'G2304', 'G2305', 'G2306', 'G2307', 'G2308', 'G2309', 'G2310', 'G2311',
        'G2312', 'G2313', 'G2314', 'G2315', 'G2316']

# define path where .mseed files are stored
path = method_path('fluvial_seismology', 'data/raw/seismic_data/NO2302/GPZ')
stas = ['ZE.2302..GPZ']  # station IDs
if not os.path.exists(path):
    os.makedirs(path, exist_ok=True)

# set station info
nowsta = stas[0]
network, station, location, channel = nowsta.split('.')
print(f"Station: {nowsta}")

# define the file pattern for reading mseed files
# assuming filenames follow the pattern 'ZE.2304..GPZ_YYYYMMDDTHHMMSS.mseed'
file_pattern = f'{path}/{network}.{station}.*.{channel}*.mseed'

# set up metadata
client = Client("IRISPH5", timeout=600) # timeout (s) - time before connection timeout is raised, default 10s
inv = client.get_stations(network=network, station=station, location=location, channel=channel, level='response')

# NRL configuration for sensor
sensors = {
    'geophone': 'sensor_DTCC_DT-SOLO_LF5_RC1850_RS43000_SG76.6_STgroundVel'  # SmartSolo sensor configuration
    # Additional sensors can be added to this dictionary as needed
}
# dictionary of EPIC loggers
loggers = {'smartsolo' : 'datalogger_DTCC_SmartSolo-IGU-16HR3C_PD18_FR1000_FPLP_DFOff', # logger for SmartSolo
           # Additional loggers can be added to this dictionary as needed
          }

# loop through each day in the date range
current_date = start_date
while current_date < end_date:
    t1 = current_date
    t2 = current_date + delta  # the next day
    print(f"\nProcessing data for {t1} to {t2}")

    try: # read data for the current day
        S = obspy.read(file_pattern, starttime=t1, endtime=t2)
        ## RSAM plot before resample?
    except Exception as e:
        print(f"Error reading data for {t1} to {t2}: {e}")
        current_date += delta
        continue  # skip to next date if there's an error reading data

    # merge traces to avoid duplicates
    S.merge(method=1)

    # or plot RSAM after merge, before resample?
    f1 = waveformUtils.multiDaySpectrogram(S, averageLength=3600, fftLength=60, minFreq=0.5, maxFreq=150, cmap='turbo',
                                      dateLimits=None,
                                      vmin=0.4, vmax=0.9, plotAverage=True)
    f1.show()

    #response_z = read_inventory(method_path('fluvial_seismology', 'data/raw/instrument_metadata/ph5-station_ZE2303GPZ.xml'))  # vertical component xml # creates an ObsPy Inventory object containing response information
    #print("Add filter...")
    #S.filter("highpass", freq=0.002).remove_response(inventory=response_z)

    # # prefiltering for instrument's frequency bandwidth, do before preprocessing
    # for tr in S: # adjust to not read below 0.1 Hz***
    #     pre_filt = [0.1, 1, 100, 150] # values TBD # 4 corner frequencies
    #     tr.remove_response(inventory=inv, pre_filt=pre_filt, plot=True) # , output="DISP", water_level=60, plot=True)

    # resample the data
    print('Resampling...')
    S.resample(175)  # Resampled rate for final analysis. Note: Consider Nyquist criterion
    ## should resample be 2*(max freq of interest) = 2*150 OR 0.5*(initial data save sample rate) = 0.5*350 ?

    # or plot RSAM after resample?

    # create PPSD object and add traces
    ppsd = None
    for tr in S:
        if ppsd is None:
            ppsd = PPSD(tr.stats, metadata=inv, period_limits=(1/150, 10.0)) # ppsd computed for 0.1 to 200 Hz --> should this be 0.0057s (175 Hz) since new resample rate??
        ppsd.add(tr)
        print('Trace added')

    # plot the PPSD for the current day
    if ppsd is not None and len(ppsd.times_processed) > 0:
        print(f"Data accumulated for {(t2-t1)/(60*60*24)}-day periods starting on {t1}")
        #print("Processed times:", ppsd.times_processed[:10])  # see first 10 times processed

        # plot the PPSD
        plt.figure(figsize=(10, 6))
        ppsd.plot(period_lim=(1/150,10),cmap=pqlx) # to zoom into a specific range, adjust period_lim
        psd_values = ppsd.psd_values
        print("Shape of psd values: ", np.shape(psd_values))
        #print(psd_values[0:1]) # first branch

        # add x-axis of PST

        print("bins: [0.008333, 0.02, 10] s = [120, 20, 0.1] Hz")
        print("bins based on spectrograms")
        # plotting time series of PSD values
        ppsd.plot_temporal([1/120,0.02,10])
        # use these values instead:
        #ppsd.plot_temporal([0.00667,0.01,0.1]) # period bins (s)

        # save the plot as .png
        #plot_filename = f"ppsd_plot_{t1.date}.png" # change title if more than one date
        #plt.savefig(plot_filename)
        #print(f"Plot saved as {plot_filename}")

        plt.close()  # close plot to avoid displaying it in each iteration

    else:
        print(f"No data accumulated for {t1} to {t2}, check if exists")

    # move to the next day
    current_date += delta

In [ ]:
#Working from documentation for Visualizing Probabalistic Power Spectra Densities in the OBSPY 1.4.1 user guide.
#Objective is to determine a strategy for assessing signal quality between different sites and times.
#Signals validated by whatever tool can be produced here are intended to be of enough quality to carry forward for figures and additional analysis

#st = read("https://examples.obspy.org/BW.KW1..EHZ.D.2011.037")
#tr = st.select(id="BW.KW1..EHZ")[0]

import obspy
from obspy import read, UTCDateTime
from obspy.io.xseed import Parser
from obspy.signal import PPSD
import numpy as np
import matplotlib.pyplot as plt
from obspy.imaging.cm import pqlx
import os
from obspy.clients.fdsn import Client
from datetime import timedelta

## example with ZE 2303 GPZ 7/13 - 7/16
## whole date range 2023-07-12 to 2023-09-08
# but exclude first and last day for cleaner data

# define start and end time for the data range
start_date = UTCDateTime(2023, 7, 13)  # Start time
end_date = UTCDateTime(2023, 7, 16)  # End time
delta = timedelta(days=1)  # define time interval between each iteration (1 day), can just use seconds instead

# define path where .mseed files are stored
path = method_path('fluvial_seismology', 'data/raw/seismic_data/NO2303')
stas = ['ZE.2303..GPZ']  # station IDs
if not os.path.exists(path):
    os.makedirs(path, exist_ok=True)

# set station info
nowsta = stas[0]
network, station, location, channel = nowsta.split('.')
print(f"Station: {nowsta}")

# define the file pattern for reading mseed files
# assuming filenames follow the pattern 'ZE.2304..GPZ_YYYYMMDDTHHMMSS.mseed'
file_pattern = f'{path}/{network}.{station}.*.{channel}*.mseed'

# set up metadata
client = Client("IRISPH5", timeout=600) # timeout (s) - time before connection timeout is raised, default 10s
inv = client.get_stations(network=network, station=station, location=location, channel=channel, level='response')

# loop through each day in the date range
current_date = start_date
while current_date < end_date:
    t1 = current_date
    t2 = current_date + delta  # The next day
    print(f"\nProcessing data for {t1} to {t2}")

    try: # read data for the current day
        S = obspy.read(file_pattern, starttime=t1, endtime=t2)
    except Exception as e:
        print(f"Error reading data for {t1} to {t2}: {e}")
        current_date += delta
        continue  # skip to next date if there's an error reading data

    # merge traces to avoid duplicates
    S.merge(method=1)

    # # prefiltering for instrument's frequency bandwidth, do before preprocessing
    # for tr in S:
    #     pre_filt = [0.01, 1, 100, 150] # values TBD # 4 corner frequencies
    #     tr.remove_response(inventory=inv, pre_filt=pre_filt, plot=True) # , output="DISP", water_level=60, plot=True)

    # resample the data
    print('Resampling...')
    S.resample(100)  # Resampled rate for final analysis. Note: Consider Nyquist criterion

    # create PPSD object and add traces
    ppsd = None
    for tr in S:
        if ppsd is None:
            ppsd = PPSD(tr.stats, metadata=inv, period_limits=(0.01, 10.0))
        ppsd.add(tr)
        print('Trace added')

    # plot the PPSD for the current day
    if ppsd is not None and len(ppsd.times_processed) > 0:
        print(f"Data accumulated for {(t2-t1)/(60*60*24)}-day periods starting on {t1}")
        print("Processed times:", ppsd.times_processed[:10])  # see first 10 times processed

        # plot the PPSD
        plt.figure(figsize=(10, 6))
        ppsd.plot(cmap=pqlx)
        psd_values = ppsd.psd_values
        print("Shape of psd values: ", np.shape(psd_values))
        #print(psd_values[0:1]) # first branch

        # plotting time series of PSD values
        ppsd.plot_temporal([0.1,1,10])

        # save the plot as .png
        #plot_filename = f"ppsd_plot_{t1.date}.png" # change title if more than one date
        #plt.savefig(plot_filename)
        #print(f"Plot saved as {plot_filename}")

        plt.close()  # close plot to avoid displaying it in each iteration

    else:
        print(f"No data accumulated for {t1} to {t2}, check if exists")

    # move to the next day
    current_date += delta